# Car Ownership Investigation

Python conversion of `src/R/car_owners_paper.rmd`.
Uses pandas, matplotlib, seaborn, and statsmodels (MNLogit) instead of tidyverse / nnet.

## Imports and configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import yaml
import pyreadstat
from pathlib import Path

import statsmodels.api as sm
from statsmodels.discrete.discrete_model import MNLogit
from scipy import stats

In [ ]:
# Locate repo root by walking up until config.yaml is found
from pathlib import Path
import yaml

_root = Path.cwd()
while not (_root / "config.yaml").exists() and _root != _root.parent:
    _root = _root.parent

with open(_root / "config.yaml") as f:
    config = yaml.safe_load(f)

data_dir = Path(config["paths"]["data_dir"])
plots_dir = _root / config["paths"].get("plots_dir", "plots")
plots_dir.mkdir(parents=True, exist_ok=True)

print(f"Repo root : {_root}")
print(f"Data dir  : {data_dir}")
print(f"Plots dir : {plots_dir}")

## Utility functions

In [ ]:
def read_clean_sheds(filepath):
    """Read SPSS file and filter out screen==3 respondents (keeps numeric codes)."""
    df, meta = pyreadstat.read_sav(str(filepath), encoding='UTF-8', apply_value_formats=False)
    df = df[df['screen'] != 3].copy()
    return df


def build_car_history_stata(all_waves_dict):
    """
    Build car history across waves with carry-forward logic.
    Replicates R's build_car_history_stata: forward-fill mob3_3 only when
    mob3_change is 0 or missing (i.e. respondent did NOT change their car).
    """
    dfs = []
    for year, df in all_waves_dict.items():
        base_cols = ['id', 'mob2_1', 'mob3_3']
        optional_cols = ['md_220', 'old', 'mob2_e', 'mob3_change', 'md_hhgr']
        cols = base_cols + [c for c in optional_cols if c in df.columns]
        subset = df[[c for c in cols if c in df.columns]].copy()
        subset['year_wave'] = int(year)

        # Recode SPSS missing codes (-2, -1) to NaN
        if 'mob3_3' in subset.columns:
            subset['mob3_3'] = pd.to_numeric(subset['mob3_3'], errors='coerce')
            subset.loc[subset['mob3_3'].isin([-2.0, -1.0]), 'mob3_3'] = np.nan

        dfs.append(subset)

    car_history = pd.concat(dfs, ignore_index=True)
    car_history = car_history.sort_values(['id', 'year_wave']).reset_index(drop=True)

    # Convert numeric columns
    for col in ['mob2_1', 'mob3_3', 'mob2_e', 'mob3_change', 'old']:
        if col in car_history.columns:
            car_history[col] = pd.to_numeric(car_history[col], errors='coerce')

    # Forward-fill mob3_3 only when mob3_change is 0 or NaN (no car change)
    def _fill_mob3(grp):
        val = grp['mob3_3'].values.copy().astype(float)
        chg = grp['mob3_change'].values if 'mob3_change' in grp.columns else np.full(len(val), np.nan)
        for i in range(1, len(val)):
            if np.isnan(val[i]) and (np.isnan(chg[i]) or chg[i] == 0):
                prev = val[:i]
                known = prev[~np.isnan(prev)]
                if len(known) > 0:
                    val[i] = known[-1]
        return pd.Series(val, index=grp.index)

    def _fill_mob2e(grp):
        if 'mob2_e' not in grp.columns:
            return pd.Series(np.nan, index=grp.index)
        val = grp['mob2_e'].values.copy().astype(float)
        chg = grp['mob3_change'].values if 'mob3_change' in grp.columns else np.full(len(val), np.nan)
        for i in range(1, len(val)):
            if np.isnan(val[i]) and (np.isnan(chg[i]) or chg[i] == 0):
                prev = val[:i]
                known = prev[~np.isnan(prev)]
                if len(known) > 0:
                    val[i] = known[-1]
        return pd.Series(val, index=grp.index)

    car_history['mob3_3_filled'] = (
        car_history.groupby('id', group_keys=False)
        .apply(_fill_mob3)
    )
    car_history['mob2_e_filled'] = (
        car_history.groupby('id', group_keys=False)
        .apply(_fill_mob2e)
    )

    return car_history


def recode_income(s):
    """Recode detailed income bands to Low / Medium / High categories."""
    mapping = {
        '\u22643,000': 'Low (\u22644,500)',
        '3,000-4,499': 'Low (\u22644,500)',
        '4,500-5,999': 'Medium (4,500-9,000)',
        '6,000-8,999': 'Medium (4,500-9,000)',
        '9,000-12,000': 'High (>9,000)',
        '>12,000': 'High (>9,000)',
    }
    return s.map(mapping)


def save_plot(fig, filename, width=12, height=8, path=None):
    """Save figure as PDF and EPS."""
    out_dir = Path(path) if path else plots_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.set_size_inches(width, height)
    fig.savefig(out_dir / f"{filename}.pdf", format='pdf', bbox_inches='tight')
    fig.savefig(out_dir / f"{filename}.eps", format='eps', bbox_inches='tight')
    print(f"Saved: {filename}.pdf and .eps")

## Load all SHEDS waves

In [ ]:
years = [2016, 2017, 2018, 2019, 2020, 2021, 2023, 2025]
waves = {}

for year in years:
    fname = config['sheds_files'][str(year)]
    fpath = data_dir / fname
    print(f"Loading {year}: {fpath}")
    waves[str(year)] = read_clean_sheds(fpath)

print("\nWave sizes:")
for yr, df in waves.items():
    print(f"  {yr}: {len(df):,} rows, {len(df.columns)} cols")

## Build car history (forward-fill engine type)

In [ ]:
car_history = build_car_history_stata(waves)

# NA recovery check for 2025
ch_2025 = car_history[car_history['year_wave'] == 2025]
original_na = ch_2025['mob3_3'].isna().sum()
filled_na   = ch_2025['mob3_3_filled'].isna().sum()
recovered   = original_na - filled_na

print(f"2025 NA recovery:")
print(f"  original_na : {original_na}")
print(f"  filled_na   : {filled_na}")
print(f"  recovered   : {recovered}")

In [ ]:
car_history_2025 = (
    car_history[car_history['year_wave'] == 2025]
    [['id', 'mob3_3_filled', 'mob2_e_filled']]
    .copy()
)
print(car_history_2025.head())
print(f"Shape: {car_history_2025.shape}")

## Build car_data: join 2025 wave with history and create derived columns

In [ ]:
w2025 = waves['2025'].copy()
car_data = w2025.merge(car_history_2025, on='id', how='left')

# --- car_owner / num_cars ---
car_data['car_owner'] = np.where(
    car_data['mob2_1'] == 0, 'No car',
    np.where(car_data['mob2_1'] >= 1, 'Car owner', None)
)
car_data['num_cars'] = car_data['mob2_1']

# --- engine_type_raw (direct observation) ---
engine_map = {1: 'Gasoline', 2: 'Diesel', 3: 'Natural gas',
              6: 'Hybrid gasoline', 7: 'Hybrid diesel', 8: 'Electric', 9: 'Other'}
car_data['engine_type_raw'] = car_data['mob3_3'].map(engine_map)

# --- engine_type (forward-filled) ---
car_data['engine_type'] = car_data['mob3_3_filled'].map(engine_map)

# --- age_group ---
age_map = {1: '18-34', 2: '35-54', 3: '55+'}
car_data['age_group'] = car_data['agegr'].map(age_map)
car_data['age_group'] = pd.Categorical(car_data['age_group'],
                                        categories=['18-34', '35-54', '55+'],
                                        ordered=True)

# --- gender ---
gender_map = {0: 'Male', 1: 'Female', 2: 'Other'}
car_data['gender'] = car_data['sex'].map(gender_map)

# --- region_label ---
region_map = {1: 'Suisse romande', 2: 'Alpen und Voralpen',
              3: 'Westmittelland', 4: 'Ostmittelland', 5: 'Ticino'}
car_data['region_label'] = car_data['region'].map(region_map)

# --- living_area ---
area_map = {1: 'City', 2: 'Agglomeration', 3: 'Countryside'}
car_data['living_area'] = car_data['md_wohntyp3_alt'].map(area_map)

# --- residence_type ---
res_map = {1: 'Tenant', 2: 'Owner'}
car_data['residence_type'] = car_data['residtype'].map(res_map)

# --- income_level ---
income_map = {1: '\u22643,000', 2: '3,000-4,499', 3: '4,500-5,999',
              4: '6,000-8,999', 5: '9,000-12,000', 6: '>12,000',
              -3: 'Prefer not to say', -1: "Don't know"}
car_data['income_level'] = car_data['seco5'].map(income_map)
income_order = ['\u22643,000', '3,000-4,499', '4,500-5,999', '6,000-8,999',
                '9,000-12,000', '>12,000', "Don't know", 'Prefer not to say']
car_data['income_level'] = pd.Categorical(car_data['income_level'],
                                           categories=income_order, ordered=True)

# --- household_size (sum of seco1b_ columns) ---
seco1b_cols = [c for c in car_data.columns if c.startswith('seco1b_')]
car_data['household_size'] = car_data[seco1b_cols].apply(
    pd.to_numeric, errors='coerce').sum(axis=1)

# --- annual_km ---
km_map = {1: '\u22645,000 km', 2: '5,001-10,000 km', 3: '10,001-15,000 km',
          4: '15,001-20,000 km', 5: '20,001-25,000 km', 6: '25,001-30,000 km',
          7: '30,001-35,000 km', 8: '35,001-40,000 km', 9: '40,001-45,000 km',
          10: '45,001-50,000 km', 11: '>50,000 km', -1: "Don't know"}
car_data['annual_km'] = car_data['mob7a'].map(km_map)

# --- distance_cat ---
def _distance_cat(v):
    if pd.isna(v):
        return np.nan
    v = int(v)
    if v in [1, 2]:
        return 'Low (\u226410,000 km/year)'
    elif v in [3, 4, 5]:
        return 'Medium (10,000-25,000 km/year)'
    elif v in [6, 7, 8, 9, 10, 11]:
        return 'High (>25,000 km/year)'
    elif v == -1:
        return 'Unknown'
    return None

car_data['distance_cat'] = car_data['mob7a'].apply(_distance_cat)
dist_order = ['Low (\u226410,000 km/year)', 'Medium (10,000-25,000 km/year)',
              'High (>25,000 km/year)', 'Unknown']
car_data['distance_cat'] = pd.Categorical(car_data['distance_cat'],
                                           categories=dist_order, ordered=True)

print(f"car_data shape: {car_data.shape}")
print(car_data[['car_owner', 'engine_type', 'age_group', 'living_area',
                'income_level', 'distance_cat']].dtypes)

## Vehicle class distribution among car owners

In [ ]:
def _vehicle_class(row):
    m = row.get('mob3_3_filled', np.nan)
    e = row.get('mob2_e_filled', np.nan)
    if m == 8 or e == 1:
        return 'Electric'
    elif m in [5, 6, 7]:
        return 'Hybrid'
    elif m == 1:
        return 'Gasoline'
    elif m == 2:
        return 'Diesel'
    elif m == 3:
        return 'Natural gas'
    elif m == 9:
        return 'Other'
    return 'Unknown/NA'

car_owners_df = car_data[car_data['car_owner'] == 'Car owner'].copy()
car_owners_df['vehicle_class'] = car_owners_df.apply(_vehicle_class, axis=1)

total_car_owners = len(car_owners_df)
print(f"Total car owners: {total_car_owners}")

vc_dist = (
    car_owners_df['vehicle_class']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'vehicle_class', 'vehicle_class': 'vehicle_class', 'count': 'n'})
)
# pandas >=2 value_counts returns column named 'count'
if 'count' in vc_dist.columns:
    vc_dist = vc_dist.rename(columns={'count': 'n'})
vc_dist['pct'] = (vc_dist['n'] / total_car_owners * 100).round(1)
print(vc_dist.to_string(index=False))

## Analysis configuration

In [ ]:
# Which demographics to include
DEMO_CONFIG = {
    'age_group':   {'include': True,  'label': 'Age Group'},
    'gender':      {'include': False, 'label': 'Gender'},
    'living_area': {'include': True,  'label': 'Living Area'},
    'income':      {'include': True,  'label': 'Income Level'},
}

# Category orders (left-to-right on bar charts)
CATEGORY_CONFIG = {
    'age_group':   {'levels': ['18-34', '35-54', '55+']},
    'gender':      {'levels': ['Male', 'Female']},
    'living_area': {'levels': ['City', 'Agglomeration', 'Countryside']},
    'income':      {'levels': ['Low (\u22644,500)', 'Medium (4,500-9,000)', 'High (>9,000)']},
}

# Regression term labels (long form for tables)
REGRESSION_LABELS = {
    'age_group':   {'age_group[T.35-54]': 'Age: 35\u201354', 'age_group[T.55+]': 'Age: 55+'},
    'gender':      {'gender[T.Female]': 'Gender: Female'},
    'living_area': {'living_area[T.Agglomeration]': 'Area: Agglomeration',
                    'living_area[T.Countryside]': 'Area: Countryside'},
    'income':      {'income_cat[T.Medium (4,500-9,000)]': 'Income: Medium (4,500\u20139,000)',
                    'income_cat[T.High (>9,000)]': 'Income: High (>9,000)'},
}

# Short labels for forest plots
REGRESSION_LABELS_SHORT = {
    'age_group':   {'age_group[T.35-54]': 'Age: 35\u201354', 'age_group[T.55+]': 'Age: 55+'},
    'gender':      {'gender[T.Female]': 'Gender: Female'},
    'living_area': {'living_area[T.Agglomeration]': 'Area: Agglomeration',
                    'living_area[T.Countryside]': 'Area: Countryside'},
    'income':      {'income_cat[T.Medium (4,500-9,000)]': 'Income: Medium',
                    'income_cat[T.High (>9,000)]': 'Income: High'},
}

INCLUDED_DEMOS = [k for k, v in DEMO_CONFIG.items() if v['include']]

print("=== Analysis Configuration ===")
for demo in INCLUDED_DEMOS:
    print(f"  \u2713 {DEMO_CONFIG[demo]['label']}")

## Demographic crosstabs for car ownership

In [ ]:
def create_crosstab(data, group_var):
    """Compute car-ownership percentage within each category of group_var."""
    sub = data.dropna(subset=[group_var, 'car_owner'])
    ct = (
        sub.groupby([group_var, 'car_owner'], observed=True)
        .size()
        .reset_index(name='n')
    )
    ct['total'] = ct.groupby(group_var, observed=True)['n'].transform('sum')
    ct['percentage'] = ct['n'] / ct['total'] * 100
    return ct


car_by_age    = create_crosstab(car_data, 'age_group')    if 'age_group'   in INCLUDED_DEMOS else None
car_by_gender = (create_crosstab(car_data, 'gender')
                 .pipe(lambda d: d[d['gender'] != 'Other'])  if 'gender' in INCLUDED_DEMOS else None)
car_by_area   = create_crosstab(car_data, 'living_area') if 'living_area' in INCLUDED_DEMOS else None

if 'income' in INCLUDED_DEMOS:
    excl = ["Don't know", 'Prefer not to say']
    car_by_income = (
        car_data
        .loc[car_data['income_level'].notna() &
             ~car_data['income_level'].isin(excl) &
             car_data['car_owner'].notna()]
        .pipe(lambda d: d.groupby(['income_level', 'car_owner'], observed=True)
              .size().reset_index(name='n'))
        .assign(total=lambda d: d.groupby('income_level', observed=True)['n'].transform('sum'))
        .assign(percentage=lambda d: d['n'] / d['total'] * 100)
    )

    income_simple_full = (
        car_by_income
        .assign(income_cat=lambda d: recode_income(d['income_level'].astype(str)))
        .dropna(subset=['income_cat'])
        .groupby(['income_cat', 'car_owner'], observed=True)
        .agg(n=('n', 'sum'), total=('total', 'sum'))
        .reset_index()
        .assign(percentage=lambda d: d['n'] / d['total'] * 100)
    )
else:
    car_by_income = None
    income_simple_full = None

print("Living area unique values:", car_by_area['living_area'].unique() if car_by_area is not None else 'N/A')

In [ ]:
# Assemble combined dataframe for the faceted stacked bar plot
parts = []
if 'age_group' in INCLUDED_DEMOS and car_by_age is not None:
    parts.append(
        car_by_age.assign(demographic=DEMO_CONFIG['age_group']['label'],
                          category=car_by_age['age_group'].astype(str))
        [['demographic', 'category', 'car_owner', 'n', 'total', 'percentage']]
    )
if 'living_area' in INCLUDED_DEMOS and car_by_area is not None:
    parts.append(
        car_by_area.assign(demographic=DEMO_CONFIG['living_area']['label'],
                           category=car_by_area['living_area'].astype(str))
        [['demographic', 'category', 'car_owner', 'n', 'total', 'percentage']]
    )
if 'gender' in INCLUDED_DEMOS and car_by_gender is not None:
    parts.append(
        car_by_gender.assign(demographic=DEMO_CONFIG['gender']['label'],
                             category=car_by_gender['gender'].astype(str))
        [['demographic', 'category', 'car_owner', 'n', 'total', 'percentage']]
    )
if 'income' in INCLUDED_DEMOS and income_simple_full is not None:
    parts.append(
        income_simple_full.assign(demographic=DEMO_CONFIG['income']['label'],
                                  category=income_simple_full['income_cat'].astype(str))
        [['demographic', 'category', 'car_owner', 'n', 'total', 'percentage']]
    )

combined_data_full = pd.concat(parts, ignore_index=True)

demo_order = [DEMO_CONFIG[d]['label'] for d in INCLUDED_DEMOS]
combined_data_full['demographic'] = pd.Categorical(
    combined_data_full['demographic'], categories=demo_order, ordered=True)

all_cat_levels = [lv for d in INCLUDED_DEMOS for lv in CATEGORY_CONFIG[d]['levels']]
combined_data_full['category'] = pd.Categorical(
    combined_data_full['category'], categories=all_cat_levels, ordered=True)
combined_data_full['car_owner'] = pd.Categorical(
    combined_data_full['car_owner'], categories=['Car owner', 'No car'], ordered=True)

combined_data_full = combined_data_full.sort_values(['demographic', 'category', 'car_owner'])
print(combined_data_full.head(10).to_string(index=False))

## Plot: Car ownership rates across demographics (stacked bar)

In [ ]:
OWNER_COLORS = {'Car owner': '#4A90A4', 'No car': '#D4634E'}

demographics_list = combined_data_full['demographic'].cat.categories.tolist()
n_facets = len(demographics_list)

fig, axes = plt.subplots(1, n_facets, figsize=(12, 8), sharey=True)
if n_facets == 1:
    axes = [axes]

for ax, demo in zip(axes, demographics_list):
    sub = (
        combined_data_full[combined_data_full['demographic'] == demo]
        .sort_values('category')
    )
    categories = sub['category'].cat.remove_unused_categories().cat.categories.tolist()

    bottom = np.zeros(len(categories))
    for owner_val in ['Car owner', 'No car']:
        vals = []
        for cat in categories:
            row = sub[(sub['category'] == cat) & (sub['car_owner'] == owner_val)]
            vals.append(row['percentage'].values[0] if len(row) > 0 else 0.0)
        vals = np.array(vals)
        bars = ax.bar(categories, vals, bottom=bottom,
                      color=OWNER_COLORS[owner_val], label=owner_val, width=0.6)
        # Annotate
        for j, (v, b) in enumerate(zip(vals, bottom)):
            if v > 3:
                ax.text(j, b + v / 2, f"{v:.1f}%",
                        ha='center', va='center', fontsize=9,
                        fontweight='bold', color='white')
        bottom += vals

    ax.set_title(demo, fontweight='bold', fontsize=13)
    ax.set_xlabel('')
    ax.set_ylim(0, 100)
    ax.set_yticks(range(0, 101, 25))
    ax.tick_params(axis='x', labelrotation=45, labelsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.grid(axis='y', color='grey', alpha=0.3, linewidth=0.5)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(False)

axes[0].set_ylabel('Percentage (%)', fontsize=13)

# Shared legend
handles = [plt.Rectangle((0, 0), 1, 1, color=OWNER_COLORS[v])
           for v in ['Car owner', 'No car']]
fig.legend(handles, ['Car owner', 'No car'],
           loc='lower center', ncol=2, fontsize=13,
           frameon=False, bbox_to_anchor=(0.5, -0.02))

fig.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()
save_plot(fig, 'car_ownership_combined_stacked', width=12, height=8)

## EV adoption by demographics

In [ ]:
def _is_electric(row):
    return row.get('engine_type') == 'Electric' or row.get('mob2_e_filled') == 1

car_owners_ev = car_data[car_data['car_owner'] == 'Car owner'].copy()
car_owners_ev['vehicle_type'] = car_owners_ev.apply(
    lambda r: 'Electric' if _is_electric(r) else 'Non-Electric', axis=1)


def _ev_crosstab(data, group_var):
    sub = data.dropna(subset=[group_var, 'engine_type'])
    ct = (
        sub.groupby([group_var, 'vehicle_type'], observed=True)
        .size()
        .reset_index(name='n')
    )
    ct['total'] = ct.groupby(group_var, observed=True)['n'].transform('sum')
    ct['percentage'] = ct['n'] / ct['total'] * 100
    return ct


ev_by_age      = _ev_crosstab(car_owners_ev, 'age_group')
ev_by_area     = _ev_crosstab(car_owners_ev, 'living_area')

ev_by_distance = (
    car_owners_ev
    .loc[car_owners_ev['distance_cat'].notna() &
         (car_owners_ev['distance_cat'] != 'Unknown') &
         car_owners_ev['engine_type'].notna()]
    .pipe(lambda d: _ev_crosstab(d, 'distance_cat'))
)

# Income EV (simplified bands with newlines replaced)
excl = ["Don't know", 'Prefer not to say']
_ev_inc = (
    car_owners_ev
    .loc[car_owners_ev['income_level'].notna() &
         ~car_owners_ev['income_level'].isin(excl) &
         car_owners_ev['engine_type'].notna()]
    .copy()
)
_ev_inc['income_cat_ev'] = _ev_inc['income_level'].astype(str).map({
    '\u22643,000': 'Low (\u22644,500)',
    '3,000-4,499': 'Low (\u22644,500)',
    '4,500-5,999': 'Medium (4,500-9,000)',
    '6,000-8,999': 'Medium (4,500-9,000)',
    '9,000-12,000': 'High (>9,000)',
    '>12,000': 'High (>9,000)',
})
_ev_inc = _ev_inc.dropna(subset=['income_cat_ev'])
ev_by_income_simple = (
    _ev_inc.groupby(['income_cat_ev', 'vehicle_type'], observed=True)
    .size()
    .reset_index(name='n')
    .assign(total=lambda d: d.groupby('income_cat_ev', observed=True)['n'].transform('sum'))
    .assign(percentage=lambda d: d['n'] / d['total'] * 100)
)
inc_ev_order = ['Low (\u22644,500)', 'Medium (4,500-9,000)', 'High (>9,000)']
ev_by_income_simple['income_cat_ev'] = pd.Categorical(
    ev_by_income_simple['income_cat_ev'], categories=inc_ev_order, ordered=True)

# EV rate by income_level (for reference)
ev_by_income = (
    car_owners_ev
    .loc[car_owners_ev['income_level'].notna() &
         ~car_owners_ev['income_level'].isin(excl) &
         car_owners_ev['engine_type'].notna()]
    .groupby(['income_level', 'vehicle_type'], observed=True)
    .size()
    .reset_index(name='n')
    .assign(percentage=lambda d: d['n'] / d.groupby('income_level', observed=True)['n'].transform('sum') * 100)
    .query("vehicle_type == 'Electric'")
)
print(ev_by_income.to_string(index=False))

print("\nev_by_income_simple income_cat_ev unique:", ev_by_income_simple['income_cat_ev'].unique().tolist())

In [ ]:
def _rename_group(df, old_col, new_name):
    return df.rename(columns={old_col: 'category'}).assign(demographic=new_name)

ev_age_part = _rename_group(
    ev_by_age[['age_group', 'vehicle_type', 'percentage']].copy(),
    'age_group', 'Age Group')

ev_area_part = _rename_group(
    ev_by_area[['living_area', 'vehicle_type', 'percentage']].copy(),
    'living_area', 'Living Area')

ev_dist_part = (
    ev_by_distance[['distance_cat', 'vehicle_type', 'percentage']]
    .copy()
    .assign(distance_cat=lambda d: d['distance_cat'].astype(str))
    .pipe(_rename_group, 'distance_cat', 'Distance Traveled')
)

ev_inc_part = (
    ev_by_income_simple[['income_cat_ev', 'vehicle_type', 'percentage']]
    .copy()
    .assign(income_cat_ev=lambda d: d['income_cat_ev'].astype(str))
    .pipe(_rename_group, 'income_cat_ev', 'Income Level')
)

combined_ev_data = (
    pd.concat([ev_age_part, ev_area_part, ev_dist_part, ev_inc_part],
              ignore_index=True)
    .query("category != 'Unknown'")
    [['demographic', 'category', 'vehicle_type', 'percentage']]
)

demo_ev_order = ['Age Group', 'Living Area', 'Distance Traveled', 'Income Level']
combined_ev_data['demographic'] = pd.Categorical(
    combined_ev_data['demographic'], categories=demo_ev_order, ordered=True)

ev_cat_levels = [
    '18-34', '35-54', '55+',
    'City', 'Agglomeration', 'Countryside',
    'Low (\u226410,000 km/year)', 'Medium (10,000-25,000 km/year)', 'High (>25,000 km/year)',
    'Low (\u22644,500)', 'Medium (4,500-9,000)', 'High (>9,000)',
]
combined_ev_data['category'] = pd.Categorical(
    combined_ev_data['category'], categories=ev_cat_levels, ordered=True)
combined_ev_data['vehicle_type'] = pd.Categorical(
    combined_ev_data['vehicle_type'], categories=['Non-Electric', 'Electric'], ordered=True)

print("NA categories:")
print(combined_ev_data[combined_ev_data['category'].isna()])
print("\nCombined EV data (first 12 rows):")
print(combined_ev_data.head(12).to_string(index=False))

## Plot: EV adoption across demographics (stacked bar)

In [ ]:
EV_COLORS = {'Electric': '#2ECC71', 'Non-Electric': '#95A5A6'}

ev_demos = combined_ev_data['demographic'].cat.categories.tolist()
n_ev_facets = len(ev_demos)

fig2, axes2 = plt.subplots(1, n_ev_facets, figsize=(12, 8), sharey=True)
if n_ev_facets == 1:
    axes2 = [axes2]

for ax, demo in zip(axes2, ev_demos):
    sub = (
        combined_ev_data[combined_ev_data['demographic'] == demo]
        .dropna(subset=['category'])
        .sort_values('category')
    )
    categories = sub['category'].cat.remove_unused_categories().cat.categories.tolist()

    bottom = np.zeros(len(categories))
    # Draw Non-Electric first (bottom), then Electric on top
    for vtype in ['Non-Electric', 'Electric']:
        vals = []
        for cat in categories:
            row = sub[(sub['category'] == cat) & (sub['vehicle_type'] == vtype)]
            vals.append(row['percentage'].values[0] if len(row) > 0 else 0.0)
        vals = np.array(vals)
        ax.bar(categories, vals, bottom=bottom,
               color=EV_COLORS[vtype], label=vtype, width=0.6)
        for j, (v, b) in enumerate(zip(vals, bottom)):
            if v > 3:
                ax.text(j, b + v / 2, f"{v:.1f}%",
                        ha='center', va='center', fontsize=8,
                        fontweight='bold', color='black')
        bottom += vals

    ax.set_title(demo, fontweight='bold', fontsize=13)
    ax.set_xlabel('')
    ax.set_ylim(0, 102)
    ax.set_yticks(range(0, 101, 25))
    ax.tick_params(axis='x', labelrotation=45, labelsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.grid(axis='y', color='grey', alpha=0.3, linewidth=0.5)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(False)

axes2[0].set_ylabel('Percentage (%)', fontsize=13)

handles2 = [plt.Rectangle((0, 0), 1, 1, color=EV_COLORS[v])
            for v in ['Electric', 'Non-Electric']]
fig2.legend(handles2, ['Electric', 'Non-Electric'],
            loc='lower center', ncol=2, fontsize=13,
            frameon=False, bbox_to_anchor=(0.5, -0.02))

fig2.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()
save_plot(fig2, 'ev_adoption_combined', width=12, height=8)

## EV adoption check: age group 35-54

In [ ]:
print(ev_by_age[ev_by_age['age_group'] == '35-54'].to_string(index=False))

## Multinomial logit: No car / Non-Electric / Electric

In [ ]:
print("car_owner distribution:")
print(car_data['car_owner'].value_counts(dropna=False))


def _vehicle_class_multi(row):
    if row.get('car_owner') == 'No car':
        return 'No car'
    et = row.get('engine_type')
    me = row.get('mob2_e_filled')
    if et == 'Electric' or me == 1:
        return 'Electric'
    if et in ['Gasoline', 'Diesel', 'Hybrid gasoline', 'Hybrid diesel', 'Natural gas', 'Other']:
        return 'Non-Electric'
    return None


def _distance_simple(v):
    if pd.isna(v):
        return np.nan
    v = int(v)
    if v in [1, 2]:
        return 'Low'
    elif v in [3, 4, 5]:
        return 'Medium'
    elif v in [6, 7, 8, 9, 10, 11]:
        return 'High'
    return None


multi_data = car_data.copy()
multi_data['vehicle_class'] = multi_data.apply(_vehicle_class_multi, axis=1)
multi_data['income_cat']     = recode_income(multi_data['income_level'].astype(str).replace('nan', None))
multi_data['distance_simple'] = multi_data['mob7a'].apply(_distance_simple)

# Filter based on DEMO_CONFIG
multi_data = multi_data.dropna(subset=['vehicle_class'])

if 'age_group' in INCLUDED_DEMOS:
    multi_data = multi_data.dropna(subset=['age_group'])
    multi_data['age_group'] = pd.Categorical(
        multi_data['age_group'],
        categories=CATEGORY_CONFIG['age_group']['levels'], ordered=True)

if 'gender' in INCLUDED_DEMOS:
    multi_data = multi_data[multi_data['gender'].notna() & (multi_data['gender'] != 'Other')]
    multi_data['gender'] = pd.Categorical(
        multi_data['gender'],
        categories=CATEGORY_CONFIG['gender']['levels'], ordered=True)

if 'living_area' in INCLUDED_DEMOS:
    multi_data = multi_data.dropna(subset=['living_area'])
    multi_data['living_area'] = pd.Categorical(
        multi_data['living_area'],
        categories=CATEGORY_CONFIG['living_area']['levels'], ordered=True)

if 'income' in INCLUDED_DEMOS:
    multi_data = multi_data.dropna(subset=['income_cat'])
    multi_data['income_cat'] = pd.Categorical(
        multi_data['income_cat'],
        categories=CATEGORY_CONFIG['income']['levels'], ordered=True)

# Outcome: ordered factor with 'No car' as baseline (index 0)
vc_order = ['No car', 'Non-Electric', 'Electric']
multi_data['vehicle_class'] = pd.Categorical(
    multi_data['vehicle_class'], categories=vc_order, ordered=True)

print("\nOutcome distribution (full sample):")
print(multi_data['vehicle_class'].value_counts().reindex(vc_order))

In [ ]:
# Build formula from included demographics
formula_terms = []
if 'age_group'   in INCLUDED_DEMOS: formula_terms.append('C(age_group, Treatment("18-34"))')
if 'gender'      in INCLUDED_DEMOS: formula_terms.append('C(gender, Treatment("Male"))')
if 'living_area' in INCLUDED_DEMOS: formula_terms.append('C(living_area, Treatment("City"))')
if 'income'      in INCLUDED_DEMOS: formula_terms.append('C(income_cat, Treatment("Low (\u22644,500)"))')

formula_str = 'vehicle_class ~ ' + ' + '.join(formula_terms)
print("Formula:", formula_str)

# Encode outcome as integer (0=No car, 1=Non-Electric, 2=Electric)
y = multi_data['vehicle_class'].cat.codes.values   # 0, 1, 2

# Design matrix via patsy
import patsy

# Use patsy to build the design matrix (endog + exog)
endog, exog = patsy.dmatrices(formula_str.replace('vehicle_class', '1') + '-1',
                               multi_data, return_type='dataframe')
# Build X manually using patsy
X = patsy.dmatrix(
    ' + '.join(formula_terms),
    multi_data, return_type='dataframe'
)

m1 = MNLogit(y, X)
result = m1.fit(method='bfgs', maxiter=1000, disp=False)
print(result.summary())

In [ ]:
# Extract tidy results (odds ratios + CIs)
# MNLogit .params uses integer positional index — access by position, not label

outcome_labels = [vc_order[i] for i in range(1, len(vc_order))]  # ['Non-Electric', 'Electric']
predictor_names = [c for c in X.columns if c != 'Intercept']
pred_positions  = {pred: X.columns.tolist().index(pred) for pred in predictor_names}

# Convert to numpy arrays to avoid index mismatch issues
params_arr = np.array(result.params)   # (n_params, n_outcomes-1)
bse_arr    = np.array(result.bse)
tval_arr   = np.array(result.tvalues)
pval_arr   = np.array(result.pvalues)

# Compute CIs from coef ± 1.96*SE (avoids conf_int() shape ambiguity across statsmodels versions)
z_crit = stats.norm.ppf(0.975)

rows = []
for col_idx, outcome in enumerate(outcome_labels):
    for pred in predictor_names:
        pos  = pred_positions[pred]
        coef = params_arr[pos, col_idx]
        se   = bse_arr[pos, col_idx]
        rows.append({
            'Outcome':  outcome,
            'term':     pred,
            'coef':     coef,
            'SE':       se,
            'z':        tval_arr[pos, col_idx],
            'p':        pval_arr[pos, col_idx],
            'OR':       np.exp(coef),
            'CI_low':   np.exp(coef - z_crit * se),
            'CI_high':  np.exp(coef + z_crit * se),
        })

m1_tbl = pd.DataFrame(rows)
m1_tbl = m1_tbl.round(3)
print(m1_tbl.to_string(index=False))

In [ ]:
# McFadden R-squared
# Null model: intercept only
X_null = np.ones((len(y), 1))
null_model = MNLogit(y, X_null)
null_result = null_model.fit(method='bfgs', maxiter=1000, disp=False)

mcfadden_r2 = 1 - result.llf / null_result.llf
print(f"McFadden R\u00b2 (Model 1): {mcfadden_r2:.3f}")

## Results table (formatted for publication)

In [ ]:
# Build combined label map from all included demographics
term_recode = {}
for demo in INCLUDED_DEMOS:
    term_recode.update(REGRESSION_LABELS.get(demo, {}))


def _sig_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    if p < 0.1:   return '.'
    return ''


table_df = (
    m1_tbl
    .query("term != 'Intercept'")
    .copy()
    .assign(
        Predictor=lambda d: d['term'].map(lambda t: term_recode.get(t, t)),
        sig=lambda d: d['p'].map(_sig_stars),
        OR_str=lambda d: (d['OR'].round(3).astype(str) + d['sig']),
        CI_str=lambda d: ('[' + d['CI_low'].round(3).astype(str)
                          + ', ' + d['CI_high'].round(3).astype(str) + ']')
    )
    [['Outcome', 'Predictor', 'OR_str', 'CI_str']]
    .rename(columns={'OR_str': 'OR', 'CI_str': '95% CI'})
)

with pd.option_context('display.max_colwidth', None, 'display.max_rows', 100):
    display(table_df)

## Forest plot: Multinomial logit odds ratios

In [ ]:
import re

def _clean_patsy_term(t):
    """Strip patsy C(varname, Treatment(...))[T.level] -> varname[T.level]."""
    m = re.match(r'C\((\w+),\s*Treatment\([^)]+\)\)\[(.+)\]', t)
    if m:
        return f'{m.group(1)}[{m.group(2)}]'
    return t

term_recode_short = {}
for demo in INCLUDED_DEMOS:
    term_recode_short.update(REGRESSION_LABELS_SHORT.get(demo, {}))

# All short labels in display order (reversed for forest plot, bottom = first)
label_order_rev = list(reversed(
    [v for demo in INCLUDED_DEMOS
     for v in REGRESSION_LABELS_SHORT.get(demo, {}).values()]
))

plot_df = (
    m1_tbl
    .query("term != 'Intercept'")
    .copy()
    .assign(
        term_clean=lambda d: d['term'].map(_clean_patsy_term),
        label=lambda d: d['term_clean'].map(lambda t: term_recode_short.get(t, t)),
        stars=lambda d: d['p'].map(_sig_stars),
    )
    .assign(
        label=lambda d: pd.Categorical(d['label'],
                                       categories=label_order_rev, ordered=True),
        Outcome=lambda d: pd.Categorical(d['Outcome'],
                                         categories=['Non-Electric', 'Electric'],
                                         ordered=True)
    )
    .dropna(subset=['label'])
    .sort_values(['label', 'Outcome'])
)

FOREST_COLORS = {'Non-Electric': '#95A5A6', 'Electric': '#2ECC71'}
outcome_vals = ['Non-Electric', 'Electric']

fig3, ax3 = plt.subplots(figsize=(9, 6))

dodge = 0.2
y_labels = label_order_rev
y_pos = {lbl: i for i, lbl in enumerate(y_labels)}

for ov, offset in zip(outcome_vals, [-dodge / 2, dodge / 2]):
    sub = plot_df[plot_df['Outcome'] == ov]
    for _, row in sub.iterrows():
        lbl = str(row['label'])
        if lbl not in y_pos:
            continue
        yy = y_pos[lbl] + offset
        ax3.errorbar(
            x=row['OR'], y=yy,
            xerr=[[row['OR'] - row['CI_low']], [row['CI_high'] - row['OR']]],
            fmt='o', color=FOREST_COLORS[ov], markersize=7,
            capsize=3, linewidth=1.8, label=ov if lbl == y_labels[0] else '')
        # Significance stars
        if row['stars']:
            ax3.text(row['CI_high'] * 1.02, yy, row['stars'],
                     va='center', ha='left', fontsize=10,
                     color=FOREST_COLORS[ov])

ax3.axvline(x=1, linestyle='--', color='grey', linewidth=1.2, alpha=0.7)
ax3.set_xscale('log')
ax3.set_xticks([0.5, 1, 2, 5, 10, 20])
ax3.get_xaxis().set_major_formatter(mticker.ScalarFormatter())
ax3.set_yticks(list(range(len(y_labels))))
ax3.set_yticklabels(y_labels, fontsize=11)
ax3.set_xlabel('')
ax3.set_ylabel('')
ax3.grid(axis='x', color='grey', alpha=0.3, linewidth=0.5)
ax3.set_axisbelow(True)
for spine in ax3.spines.values():
    spine.set_visible(False)

# Deduplicated legend
handles3, lbls3 = ax3.get_legend_handles_labels()
seen = {}
unique_h, unique_l = [], []
for h, l in zip(handles3, lbls3):
    if l not in seen and l:
        seen[l] = True
        unique_h.append(h)
        unique_l.append(l)
ax3.legend(unique_h, unique_l, loc='lower right', fontsize=11, frameon=False)

fig3.tight_layout()
plt.show()
save_plot(fig3, 'multinom_reg_car_demographics', width=9, height=6)

## Predicted probabilities by income group

In [ ]:
if 'income' in INCLUDED_DEMOS:
    income_levels = CATEGORY_CONFIG['income']['levels']

    # Build a grid holding other variables at their most common value
    def _mode(s):
        return s.mode().iloc[0] if len(s) > 0 else np.nan

    grid_rows = []
    for inc in income_levels:
        row = multi_data.copy()
        row = row.iloc[[0]].copy()  # template
        if 'age_group' in INCLUDED_DEMOS:
            row['age_group'] = _mode(multi_data['age_group'].dropna())
        if 'living_area' in INCLUDED_DEMOS:
            row['living_area'] = _mode(multi_data['living_area'].dropna())
        row['income_cat'] = inc
        grid_rows.append(row)

    grid_df = pd.concat(grid_rows, ignore_index=True)

    # Reuse the original design info so factor levels/reference categories match exactly
    X_grid = patsy.build_design_matrices([X.design_info], grid_df,
                                         return_type='dataframe')[0]

    pred_probs = result.predict(X_grid.values)  # shape (n_income, 3)
    pred_df = pd.DataFrame(pred_probs, columns=vc_order)
    pred_df.insert(0, 'income_cat', income_levels)
    pred_df = pred_df.round(3)
    print("Predicted probabilities by income group:")
    print(pred_df.to_string(index=False))

## Data quality checks

In [ ]:
print(f"Total respondents: {len(car_data)}")
print("\ncar_owner (incl. NA):")
print(car_data['car_owner'].value_counts(dropna=False))

car_owners_check = car_data[car_data['car_owner'] == 'Car owner'].copy()
print(f"\nN car owners = {len(car_owners_check)}")

n_has_engine   = car_owners_check['engine_type'].notna().sum()
n_miss_engine  = car_owners_check['engine_type'].isna().sum()
pct_miss       = n_miss_engine / len(car_owners_check) * 100

print(f"Has engine_type     : {n_has_engine}")
print(f"Missing engine_type : {n_miss_engine} ({pct_miss:.1f}% of car owners)")
print("\nengine_type breakdown:")
print(car_owners_check['engine_type'].value_counts(dropna=False))

print("\n=== Missing data for included variables ===")
car_with_engine = car_owners_check[car_owners_check['engine_type'].notna()]

if 'age_group' in INCLUDED_DEMOS:
    print(f"Missing age_group   : {car_with_engine['age_group'].isna().sum()}")
if 'gender' in INCLUDED_DEMOS:
    n_other = (car_with_engine['gender'] == 'Other').sum()
    print(f"Missing gender      : {car_with_engine['gender'].isna().sum()} | 'Other': {n_other}")
if 'living_area' in INCLUDED_DEMOS:
    print(f"Missing living_area : {car_with_engine['living_area'].isna().sum()}")
if 'income' in INCLUDED_DEMOS:
    print(f"Missing income_level: {car_with_engine['income_level'].isna().sum()}")
print(f"Missing distance (mob7a): {car_with_engine['mob7a'].isna().sum()}")